In [ ]:
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler

DIR = '/kaggle/input/llm-classification-finetuning'

# Carregar o train.csv
train_data = pd.read_csv(os.path.join(DIR, 'train.csv'))

# Todos os modelos únicos
todos_modelos = set(train_data['model_a']).union(set(train_data['model_b']))

# Inicializar contadores
vitorias = {modelo: 0 for modelo in todos_modelos}
empates = {modelo: 0 for modelo in todos_modelos}
aparicoes = {modelo: 0 for modelo in todos_modelos}

# Contabilizar vitórias, empates e aparições
for idx, row in train_data.iterrows():
    a, b = row['model_a'], row['model_b']
    aparicoes[a] += 1
    aparicoes[b] += 1
    if row['winner_model_a'] == 1:
        vitorias[a] += 1
    elif row['winner_model_b'] == 1:
        vitorias[b] += 1
    elif row['winner_tie'] == 1:
        empates[a] += 0.5
        empates[b] += 0.5

# Criar ranking
resultados = []
for modelo in todos_modelos:
    total = aparicoes[modelo]
    win = vitorias[modelo] + empates[modelo]
    taxa = win / total
    resultados.append((modelo, vitorias[modelo], empates[modelo], total, round(taxa * 100, 2)))

ranking_df = pd.DataFrame(resultados, columns=['modelo', 'vitorias', 'empates', 'aparicoes', 'win_rate'])
ranking_df.sort_values('win_rate', ascending=False, inplace=True)

# Funções auxiliares
def get_model_stats(row, win_rate_dict):
    return pd.Series({
        'model_a_win_rate': win_rate_dict.get(row['model_a'], 50),
        'model_b_win_rate': win_rate_dict.get(row['model_b'], 50),
        'win_rate_diff': win_rate_dict.get(row['model_a'], 50) - win_rate_dict.get(row['model_b'], 50)
    })

def get_vitorias(row, vitorias_dict):
    return pd.Series({
        'model_a_vitorias': vitorias_dict.get(row['model_a'], 0),
        'model_b_vitorias': vitorias_dict.get(row['model_b'], 0),
        'vitorias_diff': vitorias_dict.get(row['model_a'], 0) - vitorias_dict.get(row['model_b'], 0)
    })

def get_aparicoes(row, aparicoes_dict):
    return pd.Series({
        'model_a_aparicoes': aparicoes_dict.get(row['model_a'], 0),
        'model_b_aparicoes': aparicoes_dict.get(row['model_b'], 0),
        'aparicoes_diff': aparicoes_dict.get(row['model_a'], 0) - aparicoes_dict.get(row['model_b'], 0)
    })

# Criar features no train
win_rate_dict = ranking_df.set_index('modelo')['win_rate'].to_dict()
vitorias_dict = vitorias
aparicoes_dict = aparicoes

train_features = train_data.copy()
train_features = train_features.join(train_features.apply(lambda row: get_model_stats(row, win_rate_dict), axis=1))
train_features = train_features.join(train_features.apply(lambda row: get_vitorias(row, vitorias_dict), axis=1))
train_features = train_features.join(train_features.apply(lambda row: get_aparicoes(row, aparicoes_dict), axis=1))

# Separar binário
train_binary = train_features[train_features['winner_tie'] == 0].copy()
train_binary['target'] = train_binary['winner_model_a']

# Seleção de features
features = ['model_a_aparicoes', 'model_b_aparicoes', 'aparicoes_diff',
            'model_a_vitorias', 'model_b_vitorias', 'vitorias_diff',
            'model_a_win_rate', 'model_b_win_rate', 'win_rate_diff']

X = train_binary[features]
y = train_binary['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X.columns)

# Modelo
model = LogisticRegression(solver='liblinear', penalty='l1', C=1.0)
model.fit(X_train_scaled, y_train)

# Avaliação
y_val_pred = model.predict_proba(X_val_scaled)[:, 1]
print('Log Loss:', log_loss(y_val, y_val_pred))

# Carregar test.csv
test_data = pd.read_csv(os.path.join(DIR, 'test.csv'))

# Extrair nomes dos modelos das respostas
def extract_model_name(response):
    return response.split(':')[0].strip()

test_data['model_a'] = test_data['response_a'].apply(extract_model_name)
test_data['model_b'] = test_data['response_b'].apply(extract_model_name)

# Criar features no test (mesmo processo!)
test_features = test_data.copy()
test_features = test_features.join(test_features.apply(lambda row: get_model_stats(row, win_rate_dict), axis=1))
test_features = test_features.join(test_features.apply(lambda row: get_vitorias(row, vitorias_dict), axis=1))
test_features = test_features.join(test_features.apply(lambda row: get_aparicoes(row, aparicoes_dict), axis=1))

# Criar X_test com as MESMAS features
X_test = test_features[features]

# Escalar
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# Prever
preds = model.predict_proba(X_test_scaled)[:, 1]

# Taxa média de empate
mean_tie_rate = train_data['winner_tie'].mean()

# Submissão
submission = pd.DataFrame({
    'id': test_data['id'],
    'winner_model_a': preds * (1 - mean_tie_rate),
    'winner_model_b': (1 - preds) * (1 - mean_tie_rate),
    'winner_tie': mean_tie_rate
})

submission.to_csv('submission.csv', index=False)
print("Submissão salva como 'submission.csv'")


In [ ]:
print(train_features.dtypes)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


from scipy.stats import skew, kurtosis

print("Assimetria:", skew(X_train['win_rate_diff']))
print("Curtose:", kurtosis(X_train['win_rate_diff']))


sns.displot(X_train['win_rate_diff'], kde=True)
plt.show()
